In [5]:
import pandas as pd

df = pd.read_excel("default of credit card clients.xls", header=1)
df.to_csv("creditcard.csv", index=False)


In [6]:
df = pd.read_csv("creditcard.csv")


In [7]:
df = df.rename(columns={"default payment next month": "default"})


In [8]:
y = df["default"].values
X = df.drop(columns=["default"]).values

In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
print(X_scaled.shape, y.shape)

(30000, 24) (30000,)


In [12]:
def gaussian_noise(x, sigma=0.1):
    noise = np.random.normal(0, sigma, x.shape)
    return x + noise

def feature_dropout(x, drop_prob=0.2):
    mask = np.random.binomial(1, 1 - drop_prob, x.shape)
    return x * mask

def augment(x):
    x1 = gaussian_noise(x)
    x2 = feature_dropout(x)
    return x1, x2


In [13]:
x_sample = X_scaled[0]

view1, view2 = augment(x_sample)

print("Original:", x_sample[:5])
print("View 1:", view1[:5])
print("View 2:", view2[:5])


Original: [-1.73199307 -1.13672015  0.81016074  0.18582826 -1.05729503]
View 1: [-1.72027635 -1.07506012  0.97102326  0.17427705 -1.08760238]
View 2: [-1.73199307 -0.          0.81016074  0.18582826 -1.05729503]


In [14]:
import torch

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)


In [15]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, proj_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.projection = nn.Linear(hidden_dim, proj_dim)

    def forward(self, x):
        h = self.net(x)
        z = self.projection(h)
        z = F.normalize(z, dim=1)   # IMPORTANT
        return z


In [19]:
def nt_xent_loss(z1, z2, temperature=0.5):
    B = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2B, D)
    sim = torch.matmul(z, z.T) / temperature  # cosine similarity

    # Remove self-similarity
    mask = torch.eye(2 * B, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    # Positive similarities
    pos_sim = torch.cat([
        torch.diag(sim, B),
        torch.diag(sim, -B)
    ], dim=0)

    # NT-Xent loss
    loss = -pos_sim + torch.logsumexp(sim, dim=1)
    return loss.mean()


In [20]:
def get_batch(X, batch_size=256):
    idx = np.random.choice(len(X), batch_size, replace=False)
    x = X[idx]

    x1 = []
    x2 = []
    for sample in x:
        v1, v2 = augment(sample)
        x1.append(v1)
        x2.append(v2)

    return (
        torch.tensor(x1, dtype=torch.float32),
        torch.tensor(x2, dtype=torch.float32),
    )


In [21]:
input_dim = X_scaled.shape[1]
encoder = Encoder(input_dim)

x1, x2 = get_batch(X_scaled, batch_size=128)

z1 = encoder(x1)
z2 = encoder(x2)

loss = nt_xent_loss(z1, z2)

print("Contrastive loss:", loss.item())


Contrastive loss: 5.137942314147949


In [23]:
import torch
import torch.optim as optim

device = "cuda" #if torch.cuda.is_available() else "cpu"

input_dim = X_scaled.shape[1]
encoder = Encoder(input_dim).to(device)

optimizer = optim.Adam(encoder.parameters(), lr=1e-3)
EPOCHS = 20
BATCH_SIZE = 256


In [24]:
encoder.train()

for epoch in range(EPOCHS):
    x1, x2 = get_batch(X_scaled, batch_size=BATCH_SIZE)

    x1 = x1.to(device)
    x2 = x2.to(device)

    z1 = encoder(x1)
    z2 = encoder(x2)

    loss = nt_xent_loss(z1, z2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Contrastive Loss: {loss.item():.4f}")


Epoch 2/20 | Contrastive Loss: 5.5285
Epoch 4/20 | Contrastive Loss: 5.0907
Epoch 6/20 | Contrastive Loss: 4.8267
Epoch 8/20 | Contrastive Loss: 4.8168
Epoch 10/20 | Contrastive Loss: 4.8200
Epoch 12/20 | Contrastive Loss: 4.7917
Epoch 14/20 | Contrastive Loss: 4.7607
Epoch 16/20 | Contrastive Loss: 4.7497
Epoch 18/20 | Contrastive Loss: 4.7467
Epoch 20/20 | Contrastive Loss: 4.7402


In [25]:
torch.save(encoder.state_dict(), "tabular_contrastive_encoder.pt")

In [26]:
encoder.eval()
with torch.no_grad():
    embeddings = encoder(
        torch.tensor(X_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()

print("Embeddings shape:", embeddings.shape)


Embeddings shape: (30000, 64)


In [ ]:
from sklearn.model_selection import train_test_split

X_raw = X_scaled
X_emb = embeddings   
y_labels = y

Xr_train, Xr_test, y_train, y_test = train_test_split(
    X_raw, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

Xe_train, Xe_test, _, _ = train_test_split(
    X_emb, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

clf_raw = LogisticRegression(max_iter=1000)
clf_raw.fit(Xr_train, y_train)

pred_raw = clf_raw.predict(Xr_test)
prob_raw = clf_raw.predict_proba(Xr_test)[:, 1]

acc_raw = accuracy_score(y_test, pred_raw)
auc_raw = roc_auc_score(y_test, prob_raw)

print(f"Raw Features → Accuracy: {acc_raw:.4f}, AUC: {auc_raw:.4f}")


Raw Features → Accuracy: 0.8085, AUC: 0.7078


In [30]:
clf_emb = LogisticRegression(max_iter=1000)
clf_emb.fit(Xe_train, y_train)

pred_emb = clf_emb.predict(Xe_test)
prob_emb = clf_emb.predict_proba(Xe_test)[:, 1]

acc_emb = accuracy_score(y_test, pred_emb)
auc_emb = roc_auc_score(y_test, prob_emb)

print(f"Contrastive Embeddings → Accuracy: {acc_emb:.4f}, AUC: {auc_emb:.4f}")


Contrastive Embeddings → Accuracy: 0.8027, AUC: 0.7364
